## Classification with a Multi-Layer Perceptron

#### implement a fully-connected MLP on Pytorch, do the splitting, log the metrics blablalblab


In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

set_seed(42)

print(torch.__version__)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu") # ai suggestion 
print(device)

2.14.1
mps


In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

full_train_dataset = datasets.FashionMNIST(
    root = './data',
    train = True, 
    download = True,
    transform = transform
)

test_dataset = datasets.FashionMNIST(
    root = './data',
    train = False, 
    download = True,
    transform = transform
)

train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_train_dataset,
    [train_size, val_size],
    generator = torch.Generator().manual_seed(42)
)

In [3]:
print(len(train_dataset), len(val_dataset), len(test_dataset))

48000 12000 10000


In [4]:
full_train_dataset[0]

(tensor([[[-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000,
           -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000,
           -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000,
           -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
          [-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000,
           -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000,
           -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000,
           -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
          [-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000,
           -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000,
           -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000,
           -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
          [-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000,
           -1.0000, -1.0000, -1.000

In [5]:
full_train_dataset[0][0].shape

torch.Size([1, 28, 28])

wee here that each data point is an image, represented as tensors of pixels. (normalized)

In [6]:
full_train_dataset.classes

['T-shirt/top',
 'Trouser',
 'Pullover',
 'Dress',
 'Coat',
 'Sandal',
 'Shirt',
 'Sneaker',
 'Bag',
 'Ankle boot']

In [7]:
classes = list(full_train_dataset.classes)

classes

['T-shirt/top',
 'Trouser',
 'Pullover',
 'Dress',
 'Coat',
 'Sandal',
 'Shirt',
 'Sneaker',
 'Bag',
 'Ankle boot']

In [8]:
train_loader = DataLoader(train_dataset, batch_size = 64, shuffle = True)
val_loader = DataLoader(val_dataset, batch_size = 64, shuffle = False)
test_loader = DataLoader(test_dataset, batch_size = 64, shuffle = False)

## here we go to the architecture of the multi-layer perceptron.

each image is 28x28 resolution, so we flatten it to a vector with a length of 28 * 28 = 784.
784 -> 256 + batchnorm + relu + dropout(0.2) -> 128 + batchnorm + relu + dropout(0.2) -> 10 output

probably should consider leakyrelu in order to make it smoother

In [17]:
class FashionMLP(nn.Module):
    def __init__(self, input_dim=784, hidden_dim1=256, hidden_dim2=128, num_classes = 10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),

            nn.Linear(input_dim, hidden_dim1),
            nn.BatchNorm1d(hidden_dim1),
            nn.LeakyReLU(0.01),
            nn.Dropout(0.2),

            nn.Linear(hidden_dim1, hidden_dim2),
            nn.BatchNorm1d(hidden_dim2),
            nn.LeakyReLU(0.01),
            nn.Dropout(0.2),

            nn.Linear(hidden_dim2, num_classes)
        )

    def forward(self, x):
        return self.net(x)

model = FashionMLP().to(device)

print(model)
    

FashionMLP(
  (net): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=256, bias=True)
    (2): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (3): LeakyReLU(negative_slope=0.01)
    (4): Dropout(p=0.2, inplace=False)
    (5): Linear(in_features=256, out_features=128, bias=True)
    (6): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (7): LeakyReLU(negative_slope=0.01)
    (8): Dropout(p=0.2, inplace=False)
    (9): Linear(in_features=128, out_features=10, bias=True)
  )
)


In [13]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(outputs, dim = 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total if total > 0 else 0.0
    epoch_acc = correct / total if total > 0 else 0.0

    return epoch_loss, epoch_acc

def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            preds = torch.argmax(outputs, dim = 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    epoch_loss = running_loss / total if total > 0 else 0.0
    epoch_acc = correct / total if total > 0 else 0.0
    return epoch_loss, epoch_acc

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.001, weight_decay = 0.0001)

num_epochs = 10
history = {
    "train_loss": [],
    "val_loss": [],
    "train_acc": [],
    "val_acc": []
}

best_val_loss = float('inf')

for epoch in range(1, num_epochs + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_mlp_weights.pth')

    print(f"Epoch [{epoch:02d}/{num_epochs:02d}] | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}% | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.2f}%")

Epoch [01/10] | Train Loss: 0.5168 | Train Acc: 82.21% | Val Loss: 0.3914 | Val Acc: 85.92%


## visualization

In [ ]:
epochs_range = range(1, num_epochs + 1)

plt.figure(figsize=(14,5))

plt.plot(epochs_range, history['train_loss'], label = 'train loss', marker = 'o')
plt.plot(epochs_range, history['val_loss'], label = 'val loss', marker = 'o')
plt.title('LOSS')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.legend()
plt.grid()

plt.tight_layout()
plt.show()

In [ ]:
plt.plot(epochs_range, history['train_acc'], label = 'train accuracy', marker = 'o')
plt.plot(epochs_range, history['val_acc'], label = 'val accuracy', marker = 'o')
plt.title('ACCURACY')
plt.xlabel('epoch')
plt.ylabel('accuracy')
plt.legend()
plt.grid()

plt.tight_layout()
plt.show()

In [ ]:
model.load_state_dict(torch.load('best_mlp_weights.pth'))

test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f' Loss = {test_loss:.4f}, Accuracy = {test_acc*100:.2f}%')

images, labels = next(iter(test_loader))
model.eval()
with torch.no_grad():
    outputs = model(images.to(device))
    preds = torch.argmax(outputs, dim = 1)

images = images.cpu()
labels = labels.cpu()
preds = preds.cpu()

fig, axes = plt.subplots(1, 10, figsize=(15,3))

for i in range(10):
    ax = axes[i]
    img = images[i].squeeze() * 0.5 + 0.5
    ax.imshow(img, cmap='gray')

    true_index = labels[i].item()
    pred_index = preds[i].item()

    true_label = classes[true_index]
    pred_label = classes[pred_index]

    color = 'green' if true_label == pred_label else "red"

    ax.set_title(f'Actual: {true_label}\nPred: {pred_label}', color = color)
    ax.axis('off')


plt.tight_layout()
plt.show()

In [ ]:
model.load_state_dict(torch.load('best_mlp_weights.pth'))
model.eval()

wrong_images = []
wrong_labels = []
wrong_preds = []

with torch.no_grad():
    for images, labels in test_loader:
        images_dev = images.to(device)
        outputs = model(images_dev)
        preds = torch.argmax(outputs, dim = 1).cpu()
    
        incorrect = (preds != labels)
    
        if incorrect.any():
            wrong_images.append(images[incorrect])
            wrong_labels.append(labels[incorrect])
            wrong_preds.append(preds[incorrect])
    
        total_wrong_found = sum(len(x) for x in wrong_images)
        if total_wrong_found >= 2:
            break
            
wrong_images = torch.cat(wrong_images)
wrong_labels = torch.cat(wrong_labels)
wrong_preds = torch.cat(wrong_preds)

fig, axes = plt.subplots(1, 2, figsize=(8,4))

for i in range(2):
    ax = axes[i]
    img = wrong_images[i].squeeze() * 0.5 + 0.5
    ax.imshow(img, cmap='gray')

    true_index = wrong_labels[i].item()
    pred_index = wrong_preds[i].item()

    true_label = classes[true_index]
    pred_label = classes[pred_index]

    ax.set_title(f'Actual: {true_label}\nPred: {pred_label}', color = 'red', fontweight='bold')
    ax.axis('off')

plt.suptitle('mistaken preds', fontsize=14)
plt.tight_layout()
plt.show()